# Discovery: Resilience Rewired

Every table here comes from `discovery.py`, which is built on `engine.py` (the judges' definitions). Run top to bottom; it takes about a minute.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd, plotly.express as px
import discovery, engine
pd.set_option('display.float_format', '{:.3f}'.format)
discovery.refresh_all()

  twist_impact               0.2s
  fragile_groups             0.0s
  tipping_point              0.0s


  survival_sweep             0.9s
  regional                   0.0s
  ei_cap_effect              0.0s
  default_by_band            0.0s


  early_warning_summary      0.6s


  backtest                   0.7s


## 1. The main finding: the twist is the early warning

Each month, for every loan customer, we compute the runway the engine would have shown using only data up to that month. A customer is flagged when their runway under the EI twist (55% income) drops below 2 months.

In [2]:
print(discovery.main_finding()['headline'])
discovery.early_warning_summary()

Stress-testing every customer against the EI twist each month flags 71% of the loans that later defaulted, a median of 16 months before the default was on record. Flagged customers defaulted 8x as often (35% vs 4.2%).


,rule,label,caught,false_alarms,default_rate_flagged,default_rate_not_flagged,ratio,median_lead_months,flagged,defaults,loans
0,stress_ei_lt2,Runway under the EI twist below 2 months,0.711,0.168,0.346,0.042,8.276,15.500,156,76,682
1,base_lt3,Runway with no shock below 3 months,0.368,0.040,0.538,0.076,7.067,13.000,52,76,682
2,base_lt6,Runway with no shock below 6 months,0.566,0.117,0.377,0.058,6.492,16.000,114,76,682
3,stress_one_month_lt3,Runway after a one-month one-off below 3 months,0.842,0.335,0.240,0.029,8.290,16.000,267,76,682


## 2. Proof against real defaults (D5)

Default rate of the 682 loans by today's runway. This view is concurrent (today's buffer), so section 1 is the fair, forward-looking test.

In [3]:
db = discovery.default_by_band()
px.bar(db, x='band', y='default_rate', text='defaults', title='Default rate by runway band').show()
db

,band,default_rate,defaults,loans
0,Under 3,0.800,12,15
1,3-6,0.538,7,13
2,6-12,0.208,10,48
3,12-120,0.068,9,133
4,No shortfall / 120+,0.080,38,473


## 3. Each twist across all 4,500 customers

In [4]:
discovery.twist_impact()

,twist,under_3,under_6,out_of_money,median_runway
0,No shock,0.013,0.027,0.005,120.000
1,EI 55% for 6 months,0.097,0.351,0.006,8.600
2,One-off = a month of spending,0.063,0.087,0.041,120.000
3,EI 55% + bills +15%,0.112,0.387,0.006,7.800


## 4. Who is most fragile?

Share of customers who can't last 6 months on EI, by group.

In [5]:
fg = discovery.fragile_groups()
px.bar(fg, x='group', y='share_fail_6m_ei', color='dimension', title='Share failing 6 months on EI').show()
fg

,dimension,group,share_fail_6m_ei,customers
0,income_band,Lowest,0.044,900
1,income_band,Low,0.081,900
2,income_band,Middle,0.417,900
3,income_band,High,0.507,900
4,income_band,Highest,0.704,900
5,age_band,Under 30,0.378,1131
6,age_band,30-44,0.432,1201
7,age_band,45-59,0.431,1281
8,age_band,60+,0.089,887
9,region,Prague,0.397,554


## 5. Tipping point

Judges' formula across the bank as bills rise, with and without EI; then the C++ Monte Carlo sweep of the chance of lasting 6 months.

In [6]:
tp = discovery.tipping_point()
px.line(tp, x='bills_pct', y='under_3', color='income', title='Share under 3 months as bills rise').show()
discovery.survival_sweep()

,bills_pct,mean_survival,share_likely_fail,seconds_total,backend
0,0.000,0.644,0.358,0.880,C++ (8 threads)
1,0.050,0.633,0.368,0.880,C++ (8 threads)
2,0.100,0.622,0.382,0.880,C++ (8 threads)
3,0.150,0.611,0.392,0.880,C++ (8 threads)
4,0.200,0.600,0.404,0.880,C++ (8 threads)
5,0.250,0.589,0.412,0.880,C++ (8 threads)
6,0.300,0.578,0.424,0.880,C++ (8 threads)
7,0.350,0.567,0.434,0.880,C++ (8 threads)
8,0.400,0.556,0.444,0.880,C++ (8 threads)
9,0.450,0.545,0.456,0.880,C++ (8 threads)


## 6. Region

Does local unemployment or salary explain fragility?

In [7]:
r = discovery.regional()
print('corr with unemployment', round(r.share_fail.corr(r.unemployment), 3), '| corr with salary', round(r.share_fail.corr(r.salary), 3))
px.scatter(r, x='unemployment', y='share_fail', size='customers', hover_name='district_name', title='Districts: unemployment vs share failing 6 months on EI').show()

corr with unemployment 0.059 | corr with salary -0.045


## 7. Canadian realism (D8): the EI weekly maximum

In [8]:
e = discovery.ei_cap_effect()
print(f"Earn above the cap: {e.hit_by_cap.mean():.1%}")
print(f"Fall below 6 months only because of the cap: {((e.runway_judges >= 6) & (e.runway_capped < 6)).mean():.1%}")

Earn above the cap: 57.2%
Fall below 6 months only because of the cap: 16.1%


## 8. Is the simulation calibrated?

Profiles from data up to June 2025, simulated to December, compared with real month-end balances.

In [9]:
discovery.backtest()

,model,noise_scale,predicted_share,actual_share,brier,accounts
0,Judges' formula only (no randomness),0.000,0.027,0.018,0.028,4116
1,Full volatility,1.000,0.126,0.018,0.045,4116
2,Half volatility,0.500,0.065,0.018,0.026,4116
3,Calibrated (used),0.300,0.048,0.018,0.023,4116


## 9. Time machine for one defaulted customer

In [10]:
ew = discovery.early_warning()
acc = int(ew[(ew.defaulted == 1) & ew.lead_months.notna()].iloc[0].account_id)
tm = discovery.time_machine(acc)
px.line(tm, x='month', y=['runway_base_lt3', 'runway_stress_ei_lt2'], title=f'Account {acc}: runway the engine would have shown').update_yaxes(range=[0, 24]).show()